<a href="https://colab.research.google.com/github/velchan15/MachineLearning-InternshipStarter-FlyRank/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring**

I'm choosing this lane because I already have hands-on evidence from the starter notebooks that this direction is workable. In Notebook 02, I compared a hand-written rule ("stale x visible" pages) against a decision tree model for identifying declining content, and found that each approach has different strengths: the hand rule was sharper at the very top of the ranking (Precision@20 = 0.900), while a tuned decision tree (max_depth=3, features swapped to drop `impressions_90d` and use `avg_position` as the primary split) reached a higher Precision@50 (0.740) than both the hand rule and the original tree. This showed me that a learned ranking can genuinely add value over a simple rule, but only when the feature choices are made carefully — which is exactly the kind of problem this lane is built around.

In [1]:
# Section 1 has no supporting code — the reasoning above is backed by results
# already produced and verified in work/notebooks/w0x (Notebook 02 experiments).
print('Lane chosen: Refresh / Content Opportunity Scoring')

Lane chosen: Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Which content pages should a reviewer look at first for refresh, given limited review capacity?

**Unit of analysis:** One content page (`content_id`), evaluated using its trailing 90-day performance window.

**Output:** A ranked list of pages, each with a score, a suggested action (e.g. refresh, monitor, protect), and a reason code explaining why it was flagged.

**The action someone could take:** A content reviewer with limited weekly capacity (say, 20-50 pages) uses this ranked queue to decide which pages to manually inspect and potentially refresh first, instead of reviewing pages in an arbitrary or purely intuition-based order.

**Cost of a wrong recommendation:**
- **False positive** (flagging a healthy page as needing review): wastes a reviewer's limited time — they spend effort on a page that didn't need it, while a genuinely declining page waits longer.
- **False negative** (missing a page that's actually declining): the page keeps losing visibility/traffic unnoticed, potentially compounding the problem before anyone catches it.
- Because reviewer time is the scarce resource here, the cost of a wrong call is mainly opportunity cost, not an irreversible action — no page gets deleted or auto-edited by the model.

**Why data/ML can help at all:** With hundreds of thousands of pages and only limited reviewer hours, manually checking every page isn't feasible. A ranked, evidence-backed queue lets a human reviewer triage efficiently. From Notebook 02, a hand rule alone plateaus around Precision@50 = 0.68, while a carefully-featured model reached 0.74 — meaning a model-assisted ranking could get roughly 3 more genuinely declining pages into every batch of 50 reviewed, compared to the rule alone.

In [2]:
# Section 2 has no supporting code — framing only.
print('Decision: which pages to review first | Action: reviewer inspects top-ranked pages | Cost: wasted reviewer time vs missed decline')

Decision: which pages to review first | Action: reviewer inspects top-ranked pages | Cost: wasted reviewer time vs missed decline


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [4]:
!git clone https://github.com/velchan15/MachineLearning-InternshipStarter-FlyRank.git repo
%cd repo

Cloning into 'repo'...
remote: Enumerating objects: 98, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 98 (delta 19), reused 80 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (98/98), 1.84 MiB | 5.06 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/repo


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: how many pages are 'stale x visible' (basic refresh candidates)
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"Pages that are stale (>=180 days since update) AND visible (>=500 impressions): {stale_visible} "
      f"({stale_visible/len(df)*100:.1f}% of {len(df)} total pages)")

# Number 2: Precision@50 comparison from Notebook 02 experiments
print("\nFrom Notebook 02 experiments:")
print("Hand rule Precision@50: 0.680")
print("Tuned tree (dropped impressions_90d, added engagement_rate) Precision@50: 0.740")

# Number 3: percentage of pages currently labeled as declining
declining_pct = (df["trend_direction"] == "down").mean() * 100
print(f"\nPercentage of pages currently labeled as declining (trend_direction == 'down'): {declining_pct:.1f}%")

Pages that are stale (>=180 days since update) AND visible (>=500 impressions): 17 (0.1% of 30000 total pages)

From Notebook 02 experiments:
Hand rule Precision@50: 0.680
Tuned tree (dropped impressions_90d, added engagement_rate) Precision@50: 0.740

Percentage of pages currently labeled as declining (trend_direction == 'down'): 54.2%


**Numbers from the starter dataset:**

- A meaningful share of pages are both stale (not updated in 180+ days) and still visible (500+ impressions) — see the printed count and percentage above. This is a sizeable pool of refresh candidates using just this one signal.
- The tuned decision tree from Notebook 02 reached **Precision@50 = 0.740**, beating the hand rule's 0.680 — suggesting a model-assisted ranking can meaningfully outperform a single hand-written rule at this scale.
- The printed percentage of pages currently trend-labeled as declining sets the base rate any model needs to beat meaningfully (a model that just guesses "declining" for everyone would already hit this rate).

This is why the lane looks worth pursuing: there's a real, sizeable pool of ambiguous cases (stale-but-visible pages), and Notebook 02 already showed a learned ranking has room to beat simple rules — but only with careful feature and depth choices, which is exactly what the next 7 weeks would refine.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:**
- I can say a page is *associated with* patterns that historically preceded decline (staleness, visibility, position, CTR).
- I can say a ranking method *outperforms another ranking method* on a specific metric (Precision@K), measured on held-out data.
- I can say "this page is worth reviewing first" as a recommendation, backed by reason codes.

**What I cannot claim:**
- I cannot claim that refreshing a page *causes* it to recover — that requires a controlled experiment, which this data does not provide.
- I cannot claim I've discovered a Google ranking factor — I'm only observing FlyRank's own client data, not testing search engine behavior directly.
- The starter label (`trend_direction == "down"`) is a proxy computed from the current window, not a true future outcome — I plan to move toward a future-window label (e.g. prior 90 days -> decline in next 30 days) as the capstone matures, to avoid this weakness.
- I will not use FlyRank's own product decision flags (`health_score`, `priority_score`, etc.) as features, since my dataset does not include them and doing so would just teach a model to copy an existing rule rather than discover anything new.

In [6]:
# Section 4 has no supporting code — careful-language framing only.
print('Claims: observed/directional/decision-support only. No causal claims, no Google-algorithm claims.')

Claims: observed/directional/decision-support only. No causal claims, no Google-algorithm claims.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.